## Classic Environment Preflight

This notebook requires the classic runtime. If this check fails, rebuild with `CLASSIC=1 make notebooks-build`, restart the container, and select kernel **Python 3 (classic-langchain)**.


In [ ]:
import os
import sys

def _classic_fail(reason: str) -> None:
    raise RuntimeError(
        f"Classic runtime preflight failed: {reason}\n"
        "Fix:\n"
        "1. CLASSIC=1 make notebooks-build\n"
        "2. make notebooks-up\n"
        "3. In Jupyter, select kernel: Python 3 (classic-langchain)"
    )

kernel_name = os.getenv("JPY_KERNEL_NAME", "")
prefix = sys.prefix.lower()
if "venv-classic" not in prefix and "classic" not in kernel_name.lower():
    _classic_fail(f"detected sys.prefix={sys.prefix!r}, JPY_KERNEL_NAME={kernel_name!r}")

try:
    import langchain  # noqa: F401
except Exception as exc:
    _classic_fail(f"langchain import failed: {exc}")

print("Classic preflight passed.")


To get you even more excited as a developer: The documents don't always have to be business information, we can use these with our own openAPI specs for example. In Langchain there is a standard *APIChain* which allows us to link that kind of documentation with questions we ask the llm. 

Example from langchain:
- https://python.langchain.com/docs/use_cases/apis
> Classic track note: This notebook demonstrates legacy/classic LangChain-era patterns for evaluation and comparison.
> Prefer the modern equivalents in `lessons/2026/langchain/` for current APIs and recommended techniques.


In [ ]:
%pip install -q langchain langchain-openai

In [ ]:
%pip install -q python-dotenv
from dotenv import load_dotenv
load_dotenv()

In this example we ask the LLM to find the weather in London. We already know that the LLMs have a knowledge cut off so it can't do this. In this case we'd be asking the LLM to look for related parts in the documentation, form an api query and then provide that as extra context.

Watch the magic under the hood... No coding , just from documentation the LLM was able to construct the right calls to the API of the weather service.

In [ ]:
from _lessonshelper.pretty_print_callback_handler import PrettyPrintCallbackHandler
pretty_callback = PrettyPrintCallbackHandler()

from langchain_openai import OpenAI
llm = OpenAI(temperature=0, callbacks=[pretty_callback])


from langchain.chains.api import open_meteo_docs
from langchain.chains import APIChain
chain = APIChain.from_llm_and_api_docs(llm, open_meteo_docs.OPEN_METEO_DOCS, limit_to_domains=None)

response = chain.invoke({"question": "What is the weather like right now in London in degrees Celcius?"})
print(response["output"] if isinstance(response, dict) and "output" in response else response)